In [1]:
from pathlib import Path
import duckdb

PROJECT = Path(r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate")
DB_PATH = PROJECT / r"warehouse\realestate.duckdb"
DIR_311 = PROJECT / r"All Data Aggregated in CSV\311"

years = list(range(2015, 2027))  # 2015..2026 inclusive

con = duckdb.connect(str(DB_PATH))

loaded = []
missing = []

for y in years:
    csv_path = DIR_311 / f"{y}311.csv"   # adjust if your names differ
    if not csv_path.exists():
        # sometimes windows hides extension or your naming is slightly different
        missing.append(str(csv_path))
        continue

    con.execute(f"""
    CREATE OR REPLACE TABLE events_311_raw_{y} AS
    SELECT * FROM read_csv_auto(
      ?,
      header=true,
      delim=',',
      ignore_errors=true
    )
    """, [str(csv_path)])

    n = con.execute(f"SELECT COUNT(*) FROM events_311_raw_{y}").fetchone()[0]
    loaded.append((y, n))
    print(f"Loaded {y}: {n:,} rows")

con.close()

print("\nLoaded years:", loaded)
print("\nMissing files (check names/extensions):")
for m in missing[:10]:
    print(" -", m)
if len(missing) > 10:
    print(f" ... and {len(missing)-10} more")

Loaded 2015: 328,580 rows
Loaded 2016: 354,118 rows
Loaded 2017: 366,444 rows
Loaded 2018: 396,405 rows
Loaded 2019: 388,295 rows
Loaded 2020: 375,122 rows
Loaded 2021: 358,917 rows
Loaded 2022: 295,647 rows
Loaded 2023: 371,499 rows
Loaded 2024: 458,818 rows
Loaded 2025: 423,060 rows
Loaded 2026: 67,571 rows

Loaded years: [(2015, 328580), (2016, 354118), (2017, 366444), (2018, 396405), (2019, 388295), (2020, 375122), (2021, 358917), (2022, 295647), (2023, 371499), (2024, 458818), (2025, 423060), (2026, 67571)]

Missing files (check names/extensions):
